In [ ]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless matplotlib pandas numpy tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

In [ ]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_EPOCHS = 25
MIN_MASK_AREA_RATIO = 0.01
LEARNING_RATE = 1e-4

print("DEVICE:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAW_IMAGE_DIR = "/content/drive/MyDrive/TongueImagediabetes/data/tongue data/image"
RAW_MASK_DIR = "/content/drive/MyDrive/TongueImagediabetes/data/tongue data/label"

MODEL_SAVE_DIR = "/content/drive/MyDrive/TongueImagediabetes/seg_models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print("RAW_IMAGE_DIR :", RAW_IMAGE_DIR)
print("RAW_MASK_DIR  :", RAW_MASK_DIR)
print("MODEL_SAVE_DIR:", MODEL_SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_IMAGE_DIR : /content/drive/MyDrive/TongueImagediabetes/data/tongue data/image
RAW_MASK_DIR  : /content/drive/MyDrive/TongueImagediabetes/data/tongue data/label
MODEL_SAVE_DIR: /content/drive/MyDrive/TongueImagediabetes/seg_models


In [ ]:
def build_dataframe_from_folders(image_dir, mask_dir):
    image_dir = Path(image_dir)
    mask_dir = Path(mask_dir)

    image_paths = sorted(image_dir.glob("*.*"))
    rows = []

    for img_path in image_paths:
        mask_path = mask_dir / img_path.name

        if mask_path.exists():
            rows.append({
                "image_name": img_path.name,
                "image_path": str(img_path),
                "mask_path": str(mask_path)
            })

    return pd.DataFrame(rows)

df = build_dataframe_from_folders(RAW_IMAGE_DIR, RAW_MASK_DIR)

print("Total matched image-mask pairs:", len(df))
df.head()

Total matched image-mask pairs: 1261


,image_name,image_path,mask_path
0,1.bmp,/content/drive/MyDrive/TongueImagediabetes/dat...,/content/drive/MyDrive/TongueImagediabetes/dat...
1,1.png,/content/drive/MyDrive/TongueImagediabetes/dat...,/content/drive/MyDrive/TongueImagediabetes/dat...
2,10.0.jpg,/content/drive/MyDrive/TongueImagediabetes/dat...,/content/drive/MyDrive/TongueImagediabetes/dat...
3,10.1.jpg,/content/drive/MyDrive/TongueImagediabetes/dat...,/content/drive/MyDrive/TongueImagediabetes/dat...
4,10.2.jpg,/content/drive/MyDrive/TongueImagediabetes/dat...,/content/drive/MyDrive/TongueImagediabetes/dat...


In [ ]:
def mask_area_ratio(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return 0.0

    mask = (mask > 127).astype(np.uint8)

    area = mask.sum()
    total = mask.shape[0] * mask.shape[1]

    return area / total

original_count = len(df)

df["mask_area_ratio"] = df["mask_path"].apply(mask_area_ratio)

df = df[df["mask_area_ratio"] >= MIN_MASK_AREA_RATIO].copy()
df = df.reset_index(drop=True)

filtered_count = len(df)
removed_count = original_count - filtered_count

print("Original dataset size :", original_count)
print("Filtered dataset size :", filtered_count)
print("Removed samples       :", removed_count)

Original dataset size : 1261
Filtered dataset size : 1218
Removed samples       : 43


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))

Train samples: 974
Val samples  : 244


In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=10,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5
    ),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(),
    ToTensorV2()
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
class TongueSegmentationDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = cv2.imread(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

        # convert mask from {0,255} to {0,1}
        mask = (mask > 127).astype(np.float32)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        # add channel dimension to mask: HxW -> 1xHxW
        mask = mask.unsqueeze(0)

        return image, mask

In [ ]:
train_dataset = TongueSegmentationDataset(
    dataframe=train_df,
    transform=train_transform
)

val_dataset = TongueSegmentationDataset(
    dataframe=val_df,
    transform=val_transform
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size  :", len(val_dataset))

Train dataset size: 974
Val dataset size  : 244


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

Train batches: 122
Val batches  : 31


In [ ]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape :", masks.shape)

Image batch shape: torch.Size([8, 3, 256, 256])
Mask batch shape : torch.Size([8, 1, 256, 256])
